# Linear Discriminant Analysis (LDA) Exploration (A6)

This notebook covers the LDA implementation from scratch as part of the Python Programming Task (A6).

## Setup

In [ ]:
# Setup imports - Self-contained notebook (no external package imports)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as SklearnLDA

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

print("✅ Imports successful - Self-contained notebook")

## 1. Generate Classification Data

Create synthetic data with 3 classes and 4 features for LDA demonstration.

In [ ]:
# Generate classification data
def generate_classification_data(n_samples=200, centers=3, n_features=4, cluster_std=1.5, random_state=42):
    """Generate sample data for LDA demonstration."""
    X, y = make_blobs(n_samples=n_samples, centers=centers, n_features=n_features, 
                     cluster_std=cluster_std, random_state=random_state)
    return X, y

X, y = generate_classification_data()

print(f"Data shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Class distribution: {np.bincount(y)}")

# Standardize features (critical for LDA)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nAfter standardization:")
print(f"  Mean: {X_scaled.mean(axis=0)}")
print(f"  Std: {X_scaled.std(axis=0)}")

## 2. LDA from Scratch Implementation

Define the LDAFromScratch class directly in this notebook (self-contained).

In [ ]:
# LDA from Scratch Implementation
class LDAFromScratch:
    """Linear Discriminant Analysis implemented from scratch."""
    
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.means_ = None
        self.scalings_ = None
        self.classes_ = None
        
    def fit(self, X, y):
        """Fit LDA model to training data."""
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        
        if self.n_components is None:
            self.n_components = min(n_classes - 1, n_features)
        else:
            self.n_components = min(self.n_components, n_classes - 1, n_features)
        
        # 1. Compute class-wise means and overall mean
        self.means_ = np.array([X[y == c].mean(axis=0) for c in self.classes_])
        overall_mean = X.mean(axis=0)
        
        # 2. Construct Within-Class Scatter Matrix (Sw)
        Sw = np.zeros((n_features, n_features))
        for i, c in enumerate(self.classes_):
            X_c = X[y == c]
            diff = X_c - self.means_[i]
            Sw += diff.T @ diff
        
        # 3. Construct Between-Class Scatter Matrix (Sb)
        Sb = np.zeros((n_features, n_features))
        for i, c in enumerate(self.classes_):
            n_c = X[y == c].shape[0]
            diff = self.means_[i] - overall_mean
            Sb += n_c * np.outer(diff, diff)
        
        # 4. Solve generalized eigenvalue problem: Sw^-1 Sb
        # Use pseudo-inverse for numerical stability
        Sw_inv = np.linalg.pinv(Sw)
        eigvals, eigvecs = np.linalg.eig(Sw_inv @ Sb)
        
        # 5. Sort eigenvectors by eigenvalues (descending)
        idx = np.argsort(eigvals)[::-1]
        eigvecs = eigvecs[:, idx]
        
        # 6. Select top n_components
        self.scalings_ = eigvecs[:, :self.n_components]
        
    def transform(self, X):
        """Project data onto discriminant components."""
        return X @ self.scalings_
    
    def fit_transform(self, X, y):
        """Fit and transform in one step."""
        self.fit(X, y)
        return self.transform(X)

print("✅ LDAFromScratch class defined")

# LDA from Scratch
lda = LDAFromScratch(n_components=2)
X_lda = lda.fit_transform(X_scaled, y)

print(f"Original shape: {X.shape}")
print(f"Transformed shape: {X_lda.shape}")
print(f"Number of components: {lda.n_components}")
print(f"Classes: {lda.classes_}")

# Visualize LDA projection
def visualize_lda(X_lda, y, title, ax=None):
    """Visualize LDA projection results."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(X_lda[:, 0], X_lda[:, 1], c=y, cmap='viridis', alpha=0.6, s=50)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('LD1')
    ax.set_ylabel('LD2')
    ax.grid(True, alpha=0.3)
    return scatter

plt.figure(figsize=(8, 6))
visualize_lda(X_lda, y, 'LDA Projection (from Scratch)')
plt.colorbar(plt.gca().collections[0], label='Class')
plt.show()

## 3. Comparison with Scikit-learn LDA

Compare our from-scratch implementation with sklearn's LDA.

In [ ]:
# Compare with sklearn LDA
lda_sklearn = SklearnLDA(n_components=2)
X_lda_sklearn = lda_sklearn.fit_transform(X_scaled, y)

print(f"Sklearn LDA shape: {X_lda_sklearn.shape}")
print(f"Scratch LDA shape: {X_lda.shape}")

# Visualize side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

visualize_lda(X_lda, y, 'LDA from Scratch', ax=axes[0])
visualize_lda(X_lda_sklearn, y, 'LDA from Sklearn', ax=axes[1])

plt.tight_layout()
plt.show()

# Compare explained variance
print(f"\nSklearn explained variance ratio: {lda_sklearn.explained_variance_ratio_}")
print(f"Scratch eigenvalues (related to variance): Available in lda.scalings_")